# TCGA-LUAD DNA Methylation: Differential Methylation Analysis

Differential methylation analysis of lung adenocarcinoma (LUAD) using TCGA Illumina 450k methylation array data.

**Pipeline:** TCGA data retrieval (GDC) → beta value filtering & QC → limma differential methylation → volcano plot → top CpG heatmap

**Dataset:** TCGA-LUAD, 50 Primary Tumor + 32 Solid Tissue Normal samples, Illumina HumanMethylation450 array


## 1. Setup

Install and load required Bioconductor/CRAN packages.

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install(c(
  "TCGAbiolinks",
  "minfi",
  "limma",
  "IlluminaHumanMethylation450kanno.ilmn12.hg19",
  "IlluminaHumanMethylation450kmanifest",
  "DMRcate",
  "missMethyl",
  "sesame",
  "sesameData"
), ask = FALSE)

install.packages(c("tidyverse", "pheatmap", "ggrepel"), quiet = TRUE)


In [ ]:
library(TCGAbiolinks)
library(minfi)
library(limma)
library(IlluminaHumanMethylation450kanno.ilmn12.hg19)
library(tidyverse)

cat("All packages loaded.\n")


## 2. Data Retrieval (GDC / TCGA-LUAD)

Query the TCGA-LUAD 450k methylation cohort and select a balanced subset of tumor and normal samples.

In [ ]:
query_meth <- GDCquery(
  project       = "TCGA-LUAD",
  data.category = "DNA Methylation",
  data.type     = "Methylation Beta Value",
  platform      = "Illumina Human Methylation 450"
)

getResults(query_meth) %>% head()


In [ ]:
# Check total available samples
results <- getResults(query_meth)
cat("Total samples:", nrow(results), "\n")
table(results$sample_type)


In [ ]:
# Take 50 tumor + all 32 normal samples
tumor_ids  <- results$id[results$sample_type == "Primary Tumor"][1:50]
normal_ids <- results$id[results$sample_type == "Solid Tissue Normal"]

selected_ids <- c(tumor_ids, normal_ids)

# Subset query
query_meth_sub <- GDCquery(
  project       = "TCGA-LUAD",
  data.category = "DNA Methylation",
  data.type     = "Methylation Beta Value",
  platform      = "Illumina Human Methylation 450",
  barcode       = getResults(query_meth)$cases[getResults(query_meth)$id %in% selected_ids]
)

cat("Samples to download:", nrow(getResults(query_meth_sub)), "\n")

# Download
GDCdownload(query_meth_sub, method = "api", files.per.chunk = 10)
cat("Download done!\n")


In [ ]:
meth_data <- GDCprepare(query_meth_sub)

# Extract beta matrix
beta_matrix <- assay(meth_data)
meth_meta   <- colData(meth_data) %>% as.data.frame()

cat("Beta matrix dimensions:", dim(beta_matrix), "\n")
cat("Samples:", ncol(beta_matrix), "\n")
cat("CpG sites:", nrow(beta_matrix), "\n")
table(meth_meta$sample_type)


## 3. Filtering & QC

Remove CpGs with excessive missingness, drop sex-chromosome probes to avoid sex-related confounding, and impute remaining missing values.

In [ ]:
# Remove CpGs with NA values
na_frac    <- rowMeans(is.na(beta_matrix))
beta_clean <- beta_matrix[na_frac < 0.1, ]  # keep CpGs with <10% missing

# Remove sex chromosome CpGs (can confound analysis)
anno       <- getAnnotation(IlluminaHumanMethylation450kanno.ilmn12.hg19)
keep_cpgs  <- rownames(beta_clean) %in% rownames(anno)[!anno$chr %in% c("chrX", "chrY")]
beta_clean <- beta_clean[keep_cpgs, ]

# Impute remaining NAs with row mean
beta_clean <- t(apply(beta_clean, 1, function(x) {
  x[is.na(x)] <- mean(x, na.rm = TRUE)
  x
}))

cat("After filtering:\n")
cat("CpG sites:", nrow(beta_clean), "\n")
cat("Samples:", ncol(beta_clean), "\n")


## 4. Differential Methylation Analysis (limma)

Fit a linear model on beta values to identify CpGs differentially methylated between tumor and normal tissue.

In [ ]:
library(limma)

# Create condition factor
condition <- factor(meth_meta$sample_type, levels = c("Solid Tissue Normal", "Primary Tumor"))

# Design matrix
design <- model.matrix(~ condition)

# Fit linear model
fit <- lmFit(beta_clean, design)
fit <- eBayes(fit)

# Extract results
dmp <- topTable(fit, coef = 2, number = Inf, adjust.method = "BH")

cat("Total CpGs tested:", nrow(dmp), "\n")
cat("Hypermethylated (tumor):", sum(dmp$adj.P.Val < 0.05 & dmp$logFC > 0.2), "\n")
cat("Hypomethylated (tumor):", sum(dmp$adj.P.Val < 0.05 & dmp$logFC < -0.2), "\n")


## 5. Volcano Plot

In [ ]:
# Add significance labels
dmp$significance <- "Not significant"
dmp$significance[dmp$adj.P.Val < 0.05 & dmp$logFC >  0.2] <- "Hypermethylated"
dmp$significance[dmp$adj.P.Val < 0.05 & dmp$logFC < -0.2] <- "Hypomethylated"

# Top 10 CpGs to label
top_cpgs <- dmp %>%
  filter(significance != "Not significant") %>%
  arrange(adj.P.Val) %>%
  head(10)
top_cpgs$cpg_id <- rownames(top_cpgs)

dmp$cpg_id <- rownames(dmp)

# Plot
library(ggplot2)
library(ggrepel)

volcano_meth <- ggplot(dmp, aes(x = logFC, y = -log10(adj.P.Val), color = significance)) +
  geom_point(alpha = 0.4, size = 0.6) +
  scale_color_manual(values = c(
    "Hypermethylated" = "#E74C3C",
    "Hypomethylated"  = "#2980B9",
    "Not significant" = "grey70"
  )) +
  geom_label_repel(
    data = top_cpgs,
    aes(label = cpg_id),
    size = 3, max.overlaps = 15
  ) +
  geom_vline(xintercept = c(-0.2, 0.2), linetype = "dashed", alpha = 0.5) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", alpha = 0.5) +
  labs(
    title    = "Differential Methylation: TCGA-LUAD Tumor vs Normal",
    subtitle = "15,578 hypermethylated | 7,314 hypomethylated (|delta Beta| > 0.2, padj < 0.05)",
    x        = "Delta Beta (Tumor - Normal)",
    y        = "-log10(Adjusted P-value)",
    color    = "Methylation"
  ) +
  theme_classic(base_size = 13) +
  theme(legend.position = "right")

print(volcano_meth)

ggsave("volcano_methylation.png", volcano_meth, width = 9, height = 6, dpi = 300)


## 6. Top Differentially Methylated CpGs — Heatmap

In [ ]:
library(pheatmap)

# Get top 50 most significant CpGs
top50_cpgs  <- rownames(dmp)[1:50]
heat_matrix <- beta_clean[top50_cpgs, ]

# Annotation bar
ann_col <- data.frame(
  Condition = meth_meta$sample_type,
  row.names = colnames(heat_matrix)
)

ann_colors <- list(Condition = c(
  "Primary Tumor"       = "#E74C3C",
  "Solid Tissue Normal" = "#2980B9"
))

png("methylation_heatmap.png", width = 1000, height = 800)
pheatmap(
  heat_matrix,
  annotation_col           = ann_col,
  annotation_colors        = ann_colors,
  show_rownames            = TRUE,
  show_colnames             = FALSE,
  scale                    = "row",
  clustering_distance_rows = "euclidean",
  clustering_distance_cols = "euclidean",
  color                    = colorRampPalette(c("#2980B9", "white", "#E74C3C"))(100),
  main                     = "Top 50 Differentially Methylated CpGs -- TCGA-LUAD",
  fontsize_row             = 7
)
dev.off()

library(png)
img <- readPNG("methylation_heatmap.png")
grid::grid.raster(img)


## 7. Save Outputs

In [ ]:
write.csv(dmp, "methylation_DMP_results.csv")
saveRDS(beta_clean, "beta_clean.rds")
saveRDS(meth_meta,  "meth_metadata.rds")

cat("All result files saved.\n")


## Summary

- **485,577 → 396,164 CpG sites** retained after filtering (sex chromosomes removed, <10% missingness)
- **15,578 hypermethylated** and **7,314 hypomethylated** CpGs in tumor vs normal (padj < 0.05, |delta Beta| > 0.2)
- Top 50 differentially methylated CpGs cleanly separate tumor from normal tissue by unsupervised clustering

**Tools:** R, TCGAbiolinks, minfi, limma, pheatmap, ggplot2

**Author:** Lokaveenasri — B.Tech Biotechnology, Rajalakshmi Engineering College
